# 12 — Architecture, Ports, and Dependency Injection

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
# Provided for this lesson.
from typing import Optional

from lesson import Exercises, ___, run

check = Exercises('advanced/12', {1: '388129c84c95dd5c', 2: '9f9a0e1006024ddf', 3: '2c0c79c0a3717c80', 4: 'a053fb103df5ef32', 5: '3816425cd1eeca9e', 6: '22de6d8da9503102', 7: '4fccabf0e5097221'}, practice='advanced_01_internals_typing_architecture', independent='rename_user')

## 1. Pure domain rules

- Keep business rules in plain functions that take values and return values: no files, network or clock inside.
- Validate inputs first and raise clear errors (`ValueError` for bad values, `KeyError` for missing records).
- Return new data instead of mutating: `{**user, "name": new_name}`.

In [ ]:
def rename(user, new_name):
    if not new_name.strip():
        raise ValueError("name cannot be blank")
    return {**user, "name": new_name.strip()}

user = {"id": "u1", "name": "Ana"}
print(rename(user, "  Ann "), user)

**Exercise 1.** Return an updated copy; the original must stay the same.

In [ ]:
user = {"id": "u1", "name": "Ana"}
updated = {**user, "name": ___}
check(1, (updated["name"], user["name"]))   # rename to "Bo"

**Exercise 2.** Reject a blank name: which of these are blank after stripping?

In [ ]:
names = ["Ana", "   ", "", " Bo "]
blank = [n for n in names if not ___]   # n with the spaces stripped
check(2, blank)

## 2. Ports and adapters

- A *port* is the interface the domain needs, e.g. a `UserRepository` `Protocol` with `get` and `save`.
- An *adapter* implements it: an in-memory dict for tests, SQLite or an API in production.
- The service receives the repository as a parameter, so you can swap it without changing the rules.

In [ ]:
from typing import Protocol

class UserRepository(Protocol):
    def get(self, user_id: str) -> dict | None: ...
    def save(self, user: dict) -> None: ...

class InMemoryUserRepository:
    def __init__(self):
        self._users = {}
    def get(self, user_id):
        return self._users.get(user_id)
    def save(self, user):
        self._users[user["id"]] = dict(user)

repo = InMemoryUserRepository()
repo.save({"id": "u1", "name": "Ana"})
print(repo.get("u1"), repo.get("nope"))

**Exercise 3.** `get` returns `None` for a missing user.

In [ ]:
users = {"u1": {"id": "u1"}}
found = users.get("u2", ___)
check(3, found)

**Exercise 4.** Save stores a copy keyed by the user's id.

In [ ]:
store = {}
user = {"id": "u7", "name": "Cy"}
store[user[___]] = dict(user)
check(4, list(store))

**Exercise 5.** Does `InMemoryUserRepository` need to inherit from `UserRepository` to be accepted? Write `True` or `False`.

In [ ]:
answer = ___
check(5, answer)

## 3. Inject time

- Code that calls `datetime.now()` directly is hard to test. Take a `clock` callable instead, defaulting to `datetime.now`.
- Tests pass `lambda: fixed_time` and get the same result every run.
- Do the same for randomness (`random.Random(seed)`) and IDs.

In [ ]:
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import Callable

@dataclass
class Session:
    created_at: datetime
    ttl: timedelta
    clock: Callable[[], datetime] = datetime.now
    def is_expired(self):
        return self.clock() >= self.created_at + self.ttl

start = datetime(2024, 1, 1, 12, 0)
print(Session(start, timedelta(minutes=30), clock=lambda: start + timedelta(minutes=10)).is_expired())
print(Session(start, timedelta(minutes=30), clock=lambda: start + timedelta(hours=1)).is_expired())

**Exercise 6.** Expiry time = created plus ttl. When does a 30-minute session from 12:00 expire? Give the hour and minute.

In [ ]:
from datetime import datetime, timedelta
expires = datetime(2024, 1, 1, 12, 0) + timedelta(minutes=30)
predicted = ___   # (expires.hour, expires.minute)
check(6, predicted)

**Exercise 7.** Inject a fixed clock: is the session expired 45 minutes in?

In [ ]:
from datetime import datetime, timedelta
start = datetime(2024, 1, 1, 12, 0)
clock = lambda: start + timedelta(minutes=45)
expired = clock() >= start + timedelta(minutes=30)
predicted = ___
check(7, predicted)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `UserRepository`

The storage port: a `Protocol` with `get` and `save`.

In [ ]:
class UserRepository(Protocol):
    """Storage port: anything with get(user_id) and save(user) qualifies."""

    def get(self, user_id: str) -> Optional[dict]: ...
    def save(self, user: dict) -> None: ...

In [ ]:
check('UserRepository')

### 2. `InMemoryUserRepository`

A dict-backed repository that fits the port.

In [ ]:
class InMemoryUserRepository:
    """Satisfies UserRepository structurally (no inheritance needed)."""

    def __init__(self) -> None:
        self._users: dict[str, dict] = {}

    def get(self, user_id: str) -> Optional[dict]:
        return ___

    def save(self, user: dict) -> None:
        self._users[user["id"]] = ___

In [ ]:
check('InMemoryUserRepository')

### Independent build — `rename_user`

Using repo.get(id) and repo.save(record), return and persist the user with the stripped new name. Preserve other fields. Blank names raise ValueError; a missing user raises KeyError. Neither error saves a record.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def rename_user(repo: UserRepository, user_id: str, new_name: str) -> dict:
    raise NotImplementedError

In [ ]:
def acceptance_cases(submission):
    class Directory:
        def __init__(self):
            self.saved = []
        def get(self, user_id):
            if user_id == "u1":
                return {"id": "u1", "name": "Ada", "team": "orders"}
        def save(self, record):
            self.saved.append(record.copy())

    directory = Directory()
    updated = submission(directory, "u1", "  Lina  ")
    assert updated == {"id": "u1", "name": "Lina", "team": "orders"}, "Preserve unrelated fields and trim the name."
    assert directory.saved == [updated], "Persist the same update you return."
    for key, name, expected in [("u1", "  ", ValueError), ("absent", "Lina", KeyError)]:
        try:
            submission(directory, key, name)
        except expected:
            pass
        else:
            raise AssertionError("Check the failure behavior for " + repr((key, name)))
    assert directory.saved == [updated], "A rejected rename must not save anything."

check.assess('rename_user', acceptance_cases)

### 4. `Session`

A session whose `is_expired()` uses an injected clock.

In [ ]:
class Session:
    """A login session. is_expired() uses an injected clock so tests control time."""
    created_at: datetime
    ttl: timedelta
    clock: Callable[[], datetime] = datetime.now

    def is_expired(self) -> bool:
        return ___

In [ ]:
check('Session')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** In ports and adapters, the domain code should…

   - **a)** Depend on small interfaces (ports), not concrete I/O
   - **b)** Call the database directly
   - **c)** Read environment variables everywhere

**2.** Injecting a clock lets you…

   - **a)** Test time-dependent logic deterministically
   - **b)** Avoid datetime
   - **c)** Make code faster

**3.** A fake repository in tests is…

   - **a)** Production code
   - **b)** A simple in-memory implementation of the port
   - **c)** A mocked database server

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/12', {1: '73bb11f6f67a', 2: '99925a6649ad', 3: '944ad110d95b'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [13 — Security: Validation, Safe Parsing, Secrets and Dependencies](13_security_safe_parsing_secrets.ipynb)